In [1]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
import pyarrow as pa

In [5]:
cd parquet

[Errno 2] No such file or directory: 'parquet'
/Users/chemasj/Proyectos/SerialFormats/parquet


/Users/chemasj/.pyenv/versions/3.11.9/lib/python3.11/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


In [6]:
applications = pd.read_csv('application_record.csv')

In [7]:
applications = pd.concat(10*[applications]).reset_index().drop(columns=['ID','index']).reset_index().rename(columns={'index':'ID'})

In [8]:
applications.head(5).T

,0,1,2,3,4
ID,0,1,2,3,4
CODE_GENDER,M,M,M,F,F
FLAG_OWN_CAR,Y,Y,Y,N,N
FLAG_OWN_REALTY,Y,Y,Y,Y,Y
CNT_CHILDREN,0,0,0,0,0
AMT_INCOME_TOTAL,427500.0,427500.0,112500.0,270000.0,270000.0
NAME_INCOME_TYPE,Working,Working,Working,Commercial associate,Commercial associate
NAME_EDUCATION_TYPE,Higher education,Higher education,Secondary / secondary special,Secondary / secondary special,Secondary / secondary special
NAME_FAMILY_STATUS,Civil marriage,Civil marriage,Married,Single / not married,Single / not married
NAME_HOUSING_TYPE,Rented apartment,Rented apartment,House / apartment,House / apartment,House / apartment


In [9]:
applications['MONTH_INCOME_TOTAL'] = applications['AMT_INCOME_TOTAL']/12
applications['AGE'] = - np.floor(applications['DAYS_BIRTH']/365)

In [10]:
my_schema = pa.Schema.from_pandas(applications)
my_schema

ID: int64
CODE_GENDER: string
FLAG_OWN_CAR: string
FLAG_OWN_REALTY: string
CNT_CHILDREN: int64
AMT_INCOME_TOTAL: double
NAME_INCOME_TYPE: string
NAME_EDUCATION_TYPE: string
NAME_FAMILY_STATUS: string
NAME_HOUSING_TYPE: string
DAYS_BIRTH: int64
DAYS_EMPLOYED: int64
FLAG_MOBIL: int64
FLAG_WORK_PHONE: int64
FLAG_PHONE: int64
FLAG_EMAIL: int64
OCCUPATION_TYPE: string
CNT_FAM_MEMBERS: double
MONTH_INCOME_TOTAL: double
AGE: double
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 2759

In [11]:
my_schema = my_schema.set(12, pa.field('FLAG_MOBIL', 'bool'))
my_schema = my_schema.set(13, pa.field('FLAG_WORK_PHONE', 'bool'))
my_schema = my_schema.set(14, pa.field('FLAG_PHONE', 'bool'))
my_schema = my_schema.set(15, pa.field('FLAG_EMAIL', 'bool'))
my_schema = my_schema.remove(10)

In [12]:
%%time
applications.to_parquet('applications_processed.parquet', schema = my_schema)


CPU times: user 1.72 s, sys: 93.8 ms, total: 1.81 s
Wall time: 1.83 s


In [13]:
%%time
applications.to_csv('applications_processed.csv')

CPU times: user 15 s, sys: 421 ms, total: 15.4 s
Wall time: 15.7 s


In [14]:
applications.to_parquet('APPLICATIONS_PROCESSED', schema = my_schema, partition_cols=['NAME_INCOME_TYPE'])


In [15]:
%%time
test = pd.read_parquet('applications_processed.parquet')
test[test.NAME_INCOME_TYPE=='Working']

CPU times: user 1.17 s, sys: 266 ms, total: 1.44 s
Wall time: 685 ms


,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,MONTH_INCOME_TOTAL,AGE
0,0,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-4542,True,True,False,False,None,2.0,35625.0,33.0
1,1,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-4542,True,True,False,False,None,2.0,35625.0,33.0
2,2,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-1134,True,False,False,False,Security staff,2.0,9375.0,59.0
10,10,M,Y,Y,0,270000.0,Working,Higher education,Married,House / apartment,-769,True,True,True,True,Accountants,2.0,22500.0,47.0
11,11,M,Y,Y,0,270000.0,Working,Higher education,Married,House / apartment,-769,True,True,True,True,Accountants,2.0,22500.0,47.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4385555,4385555,M,Y,Y,1,355050.0,Working,Secondary / secondary special,Married,House / apartment,-2614,True,False,False,False,None,3.0,29587.5,44.0
4385556,4385556,M,Y,Y,1,355050.0,Working,Secondary / secondary special,Married,House / apartment,-2614,True,False,False,False,None,3.0,29587.5,44.0
4385561,4385561,M,Y,Y,1,135000.0,Working,Secondary / secondary special,Married,House / apartment,-2095,True,False,False,False,Laborers,3.0,11250.0,35.0
4385566,4385566,F,N,N,0,103500.0,Working,Secondary / secondary special,Single / not married,House / apartment,-3007,True,False,False,False,Laborers,1.0,8625.0,44.0


In [16]:
%%time
pd.read_parquet('APPLICATIONS_PROCESSED/NAME_INCOME_TYPE=Working/')
# OR (the run-time below corresponds to either one of way of reading)
pd.read_parquet('APPLICATIONS_PROCESSED', filters=[('NAME_INCOME_TYPE', '=', 'Working')])


CPU times: user 2.21 s, sys: 337 ms, total: 2.54 s
Wall time: 811 ms


,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,MONTH_INCOME_TOTAL,AGE,NAME_INCOME_TYPE
0,0,M,Y,Y,0,427500.0,Higher education,Civil marriage,Rented apartment,-4542,True,True,False,False,None,2.0,35625.0,33.0,Working
1,1,M,Y,Y,0,427500.0,Higher education,Civil marriage,Rented apartment,-4542,True,True,False,False,None,2.0,35625.0,33.0,Working
2,2,M,Y,Y,0,112500.0,Secondary / secondary special,Married,House / apartment,-1134,True,False,False,False,Security staff,2.0,9375.0,59.0,Working
3,10,M,Y,Y,0,270000.0,Higher education,Married,House / apartment,-769,True,True,True,True,Accountants,2.0,22500.0,47.0,Working
4,11,M,Y,Y,0,270000.0,Higher education,Married,House / apartment,-769,True,True,True,True,Accountants,2.0,22500.0,47.0,Working
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4522075,4385555,M,Y,Y,1,355050.0,Secondary / secondary special,Married,House / apartment,-2614,True,False,False,False,None,3.0,29587.5,44.0,Working
4522076,4385556,M,Y,Y,1,355050.0,Secondary / secondary special,Married,House / apartment,-2614,True,False,False,False,None,3.0,29587.5,44.0,Working
4522077,4385561,M,Y,Y,1,135000.0,Secondary / secondary special,Married,House / apartment,-2095,True,False,False,False,Laborers,3.0,11250.0,35.0,Working
4522078,4385566,F,N,N,0,103500.0,Secondary / secondary special,Single / not married,House / apartment,-3007,True,False,False,False,Laborers,1.0,8625.0,44.0,Working


In [17]:
filters = [('NAME_INCOME_TYPE', 'in', ['Working', 'State servant'])]
pd.read_parquet('APPLICATIONS_PROCESSED', filters=filters)

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,MONTH_INCOME_TOTAL,AGE,NAME_INCOME_TYPE
0,62,F,N,Y,1,211500.0,Secondary / secondary special,Civil marriage,House / apartment,-7099,True,False,False,False,Core staff,3.0,17625.0,45.0,State servant
1,63,F,N,Y,1,211500.0,Secondary / secondary special,Civil marriage,House / apartment,-7099,True,False,False,False,Core staff,3.0,17625.0,45.0,State servant
2,167,M,Y,Y,0,112500.0,Secondary / secondary special,Married,House / apartment,-2381,True,False,True,False,Drivers,2.0,9375.0,58.0,State servant
3,168,M,Y,Y,0,112500.0,Secondary / secondary special,Married,House / apartment,-2381,True,False,True,False,Drivers,2.0,9375.0,58.0,State servant
4,169,M,Y,Y,0,112500.0,Secondary / secondary special,Married,House / apartment,-2381,True,False,True,False,Drivers,2.0,9375.0,58.0,State servant
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5245795,4385555,M,Y,Y,1,355050.0,Secondary / secondary special,Married,House / apartment,-2614,True,False,False,False,None,3.0,29587.5,44.0,Working
5245796,4385556,M,Y,Y,1,355050.0,Secondary / secondary special,Married,House / apartment,-2614,True,False,False,False,None,3.0,29587.5,44.0,Working
5245797,4385561,M,Y,Y,1,135000.0,Secondary / secondary special,Married,House / apartment,-2095,True,False,False,False,Laborers,3.0,11250.0,35.0,Working
5245798,4385566,F,N,N,0,103500.0,Secondary / secondary special,Single / not married,House / apartment,-3007,True,False,False,False,Laborers,1.0,8625.0,44.0,Working


In [18]:
pd.read_parquet('APPLICATIONS_PROCESSED')

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,MONTH_INCOME_TOTAL,AGE,NAME_INCOME_TYPE
0,3,F,N,Y,0,270000.0,Secondary / secondary special,Single / not married,House / apartment,-3051,True,False,True,True,Sales staff,1.0,22500.0,53.0,Commercial associate
1,4,F,N,Y,0,270000.0,Secondary / secondary special,Single / not married,House / apartment,-3051,True,False,True,True,Sales staff,1.0,22500.0,53.0,Commercial associate
2,5,F,N,Y,0,270000.0,Secondary / secondary special,Single / not married,House / apartment,-3051,True,False,True,True,Sales staff,1.0,22500.0,53.0,Commercial associate
3,6,F,N,Y,0,270000.0,Secondary / secondary special,Single / not married,House / apartment,-3051,True,False,True,True,Sales staff,1.0,22500.0,53.0,Commercial associate
4,13,M,Y,Y,0,135000.0,Secondary / secondary special,Married,House / apartment,-1194,True,False,False,False,Laborers,2.0,11250.0,49.0,Commercial associate
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8771135,4385555,M,Y,Y,1,355050.0,Secondary / secondary special,Married,House / apartment,-2614,True,False,False,False,None,3.0,29587.5,44.0,Working
8771136,4385556,M,Y,Y,1,355050.0,Secondary / secondary special,Married,House / apartment,-2614,True,False,False,False,None,3.0,29587.5,44.0,Working
8771137,4385561,M,Y,Y,1,135000.0,Secondary / secondary special,Married,House / apartment,-2095,True,False,False,False,Laborers,3.0,11250.0,35.0,Working
8771138,4385566,F,N,N,0,103500.0,Secondary / secondary special,Single / not married,House / apartment,-3007,True,False,False,False,Laborers,1.0,8625.0,44.0,Working
